In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import matplotlib.pyplot as plt
import scienceplots

import time
import math
import os
import sys
import random
from functools import partial
from decimal import Decimal
import numpy as np
import scipy.io as sio
import pysindy as ps
from tqdm import trange

sys.path.insert(0, '../')
from utils import *
from solvel0 import solvel0, MIOSR
from best_subset import backward_refinement, brute_force_all_subsets, brute_force
from UBIC import *
from bayesian_model_evidence import log_evidence

from skimage.restoration import estimate_sigma
import bm3d
from kneed import KneeLocator

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel

from rdata import read_rds
from selective_inference import forward_stop_rule

from sklearn.preprocessing import StandardScaler
from sklearn import covariance
from sklearn.linear_model import (LinearRegression, Ridge, BayesianRidge, 
                                    Lasso, lars_path, ElasticNet, MultiTaskLasso, MultiTaskElasticNet, ARDRegression)
from abess import LinearRegression as AbessLinearRegression
from knockpy import KnockoffFilter, knockoff_stats, knockoffs
from knockpy.utilities import estimate_covariance
from scipy import stats
from statsmodels.stats.multitest import multipletests
from c2st.check import c2st # https://github.com/psteinb/c2st

from mbic import mbic, mbic2, ebic

from selective_inference import sfs_si, stepwise_selective_inference, subset_fdr
import fpsample
from dppy.finite_dpps import FiniteDPP

from si4pipeline import (
                        construct_pipelines, 
                        extract_features, 
                        initialize_dataset, 
                        intersection, 
                        lasso, 
                        marginal_screening, 
                        stepwise_feature_selection, 
                        union, 
                        PipelineManager
                        )

from pymcdm import weights as obj_w
from compromise_programming import optimal_decision, compromise_programming, mcdm
from pyRankMCDA.algorithm import rank_aggregation

In [ ]:
target_name = 'u'
X_pre = np.load("../Cache/X_pre_GS_2025.npy")
uv_pre = np.load("../Cache/y_pre_GS_2025.npy")

### Ground truth ###
ground_indices_u = (0, 1, 8, 12, 18, 26)
ground_coeff_u = np.array([0.014, -0.014, -1.000, 0.020, 0.020, 0.020])
ground_indices_v = (2, 8, 13, 19, 27)
ground_coeff_v = np.array([-0.067, 1.0, 0.01, 0.01, 0.01])
feature_names = np.load("../Cache/feature_names_GS_2025.npy", allow_pickle=True)
# R
fsInf = read_rds(f"../R/R_data/fsInf_screening_GS_{target_name}.rds")

X_pre_top = StandardScaler(with_std=True).fit_transform(X_pre)
uv_pre = StandardScaler(with_std=False).fit_transform(uv_pre)
if target_name == 'u':
    y_pre = uv_pre[:, 0:1]
elif target_name == 'v':
    y_pre = uv_pre[:, 1:2]
else:
    raise ValueError("target_name is either 'u' or 'v'.")

### s_max

In [ ]:
from skglm import MCPRegression, Lasso
from skglm.experimental import SqrtLasso
from scipy.stats import norm
# statsmodels
from scipy.special import comb
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_val_score

def lambda_max(X, y):
    """
    Compute lambda_max for MCP (same as Lasso): max abs correlation / n.
    Assumes X standardized, y centered.
    """
    n, p = X.shape
    return np.max(np.abs(X.T @ y)) / n

# c > 0
def plugin_lasso_lambda(X, y, c=0.25, t=np.finfo(np.float32).tiny, homoskedastic=True):
    n, p = X.shape
    assert n == len(y)

    lambda_value = 4*np.sqrt((t**2 + 2 * np.log(p)) / n)

    if homoskedastic:
        ols_result = sm.OLS(y, X).fit()
        # var = y.T.dot(y) / n
        var = np.sum(ols_result.resid**2) / ols_result.df_resid
        sigma = np.sqrt(var)
        lambda_value = sigma * lambda_value

    return float(c*lambda_value)

def rebic(y_true, y_pred, n_params, n_features, gamma=1, robust=False, exact=True, sic=False):
    if y_true.shape != y_pred.shape:
        y_true = y_true.reshape(y_pred.shape)
    n_samples = len(y_true)
    rss = np.sum((y_true - y_pred) ** 2)
    bic = n_samples*np.log(rss/n_samples) + n_params*np.log(n_samples)
    if sic:
        return bic    
    rss0 = (np.linalg.norm(y_true, ord=2)**2)/n_samples
    if exact:
        ebic = bic + 2*gamma*np.log(comb(n_features, n_params))
        rebic = n_samples*np.log(rss/n_samples) + n_params*np.log(n_samples/(2*np.pi)) + (n_params+2)*np.log(rss0/rss) + 2*gamma*np.log(comb(n_features, n_params))
    else:
        ebic = bic + 2*n_params*gamma*np.log(n_features)
        rebic = n_samples*np.log(rss/n_samples) + n_params*np.log(n_samples/(2*np.pi)) + (n_params+2)*np.log(rss0/rss) + 2*n_params*gamma*np.log(n_features)
    if not robust:
        return ebic
    return rebic

def rebic_scorer(est, X, y, gamma=1, robust=False, exact=True, sic=False):
    pred = est.predict(X)
    assert pred.shape == y.shape
    n_samples, n_tasks = pred.shape
    n_params = np.count_nonzero(est.coef_) // n_tasks
    n_features = np.prod(est.coef_.shape) // n_tasks
    return np.mean([rebic(y[:, i], pred[:, i], n_params, n_features, 
                          gamma=gamma, robust=robust, exact=exact, sic=sic) for i in range(n_tasks)])

def penalized_mse_scorer(est, X, y, l0_penalty=None):
    pred = est.predict(X)
    if y.shape != pred.shape:
        y = y.reshape(pred.shape)
    rss = np.linalg.norm(y - est.predict(X), 2)
    n_params = np.count_nonzero(est.coef_) + (1 if est.fit_intercept else 0)
    if l0_penalty is None:
        l0_penalty = 1e-3 * np.linalg.cond(X)
    return rss + l0_penalty * n_params

cv = 3
l1_mse = 1; l1_ic = 1
mse_min = np.inf; ic_min = np.inf
# penalized_mse_scorer = partial(penalized_mse_scorer, l0_penalty=None)
penalized_mse_scorer = partial(penalized_mse_scorer, l0_penalty=10**sci_format(min(np.linalg.lstsq(X_pre_top, uv_pre, rcond=None)[1]))[1])
rebic_scorer = partial(rebic_scorer, gamma=1, robust=False, sic=False)

for c in (0.5, 0.75, 1):
    est_lambda = min(plugin_lasso_lambda(X_pre_top, uv_pre[:, 0], c=c, t=1e-6, homoskedastic=True),
                     plugin_lasso_lambda(X_pre_top, uv_pre[:, 1], c=c, t=1e-6, homoskedastic=True))
    multitask_lasso = MultiTaskElasticNet(alpha=est_lambda, fit_intercept=False, max_iter=2000)
    avg_mse_score = cross_val_score(multitask_lasso, X_pre_top, uv_pre, cv=cv, scoring=penalized_mse_scorer).mean()
    avg_ic_score = cross_val_score(multitask_lasso, X_pre_top, uv_pre, cv=cv, scoring=rebic_scorer).mean()
    if avg_mse_score < mse_min:
        mse_min = avg_mse_score
        l1_mse = c
    if avg_ic_score < ic_min:
        ic_min = avg_ic_score
        l1_ic = c
    print(c, avg_mse_score, avg_ic_score)

l1_c = max(l1_mse, l1_ic)
est_lambda = min(plugin_lasso_lambda(X_pre_top, uv_pre[:, 0], c=l1_c, t=1e-6, homoskedastic=True), 
                 plugin_lasso_lambda(X_pre_top, uv_pre[:, 1], c=l1_c, t=1e-6, homoskedastic=True))
multitask_lasso = MultiTaskElasticNet(alpha=est_lambda, fit_intercept=False, max_iter=2000).fit(X_pre_top, uv_pre)
s_max = np.count_nonzero(multitask_lasso.coef_)//2
l1_c, s_max

### Knockoffs

In [ ]:
from hidimstat import (model_x_knockoff, 
                    model_x_knockoff_pvalue, 
                    model_x_knockoff_bootstrap_quantile, 
                    model_x_knockoff_bootstrap_e_value)

from sklearn.preprocessing import StandardScaler
from sklearn import covariance
from abess import LinearRegression as AbessLinearRegression
from knockpy import KnockoffFilter, knockoff_stats, knockoffs
from knockpy.utilities import estimate_covariance
from scipy import stats
from statsmodels.stats.multitest import multipletests
from c2st.check import c2st # https://github.com/psteinb/c2st
    
print(s_max)
importance_type = 'swap'
lr = AbessLinearRegression(path_type='gs', s_max=s_max, fit_intercept=False, cv=3, screening_size=0)
if importance_type == 'shap':
    kfilter = KnockoffFilter(ksampler='gaussian', fstat=knockoff_stats.ShapStatistic(model=lr), knockoff_kwargs={'method':'ci'}) # ci, spd
elif importance_type == 'lasso':
    kfilter = KnockoffFilter(ksampler='gaussian', fstat='lasso', knockoff_kwargs={'method':'ci'})
else:
    kfilter = KnockoffFilter(ksampler='gaussian', fstat=knockoff_stats.FeatureStatistic(model=lr), 
                             knockoff_kwargs={'method':'ci'}, fstat_kwargs={'feature_importance': importance_type}) # swap or swapint

fdr = 1/3 # 1/5, 1/4, 1/3, 1/2
shrinkage = "ledoitwolf" # "ledoitwolf", "MLE"
shrinkage_tol = 1e-2
rejections = []
test_scores = []
thresholds = []
for _ in trange(50):
    # tol = 1e-2, 1e-3, 1e-4
    # rejection = kfilter.forward(X=X_pre_top, y=y_pre.flatten(), fdr=fdr, shrinkage=shrinkage, recycle_up_to=0.5, tol=shrinkage_tol)
    rejection = kfilter.forward(X=X_pre_top, y=(y_pre/y_pre.std()).flatten(), fdr=fdr, shrinkage=shrinkage, recycle_up_to=0.5, tol=shrinkage_tol)
    rejection = sorted(set(np.where(rejection == 1)[0]))
    if len(rejection) > 0:
        rejections.append(rejection)
        test_scores.append(kfilter.W)
        thresholds.append(kfilter.threshold)

aggregated_ko_selection, _, _ = model_x_knockoff_bootstrap_quantile(test_scores, 
                                                                    fdr=fdr, gamma=0.5, 
                                                                    fdr_control='bhq')
print("model_x_knockoff_bootstrap_quantile:")
print(feature_names[aggregated_ko_selection])

# ShapStatistic, u: ['1' 'x0' 'x0 x1^2' 'x0_33' 'x0_22' 'x1_12' 'x0_11']
# ShapStatistic, v: ['x1' 'x0 x1^2' 'x1_33' 'x1_22' 'x1_11']
# swap, u: ['1' 'x0' 'x0 x1^2' 'x0_33' 'x0_22' 'x1_12' 'x0_11']
# swap, v: ['x1' 'x0 x1^2' 'x1_33' 'x1_22' 'x1_11']
# swapint, u: ['1' 'x0' 'x0 x1^2' 'x0_33' 'x0_22' 'x1_12' 'x0_11']
# swapint, v: ['x1' 'x0 x1^2' 'x1_33' 'x1_22' 'x1_11']
eval_selection = []
while True:
    eval_selection, _, _ = model_x_knockoff_bootstrap_e_value(test_scores, thresholds, fdr=fdr)
    if len(eval_selection) > 0:
        break
    else:
        fdr += 1e-3
print("model_x_knockoff_bootstrap_e_value:")
print(feature_names[eval_selection])
print(fdr)

In [ ]:
def shap_feature_elimination(X, y, threshold=0.01):
    threshold = min(threshold, 1)
    keep = np.arange(X.shape[1])
    remove = []
    while True:
        shap_values = shap_linear_importance(X[:, keep], y, scale=False, full=True)
        abs_shap = np.abs(shap_values).mean(axis=0)
        abs_shap_pct = (abs_shap / abs_shap.sum())
        keep_mask = abs_shap_pct >= threshold
        if keep_mask.all():
            break
        remove.extend(keep[~keep_mask])
        keep = keep[keep_mask]
    return keep.tolist(), remove

rejections = np.array(eval_selection)
assert len(rejections) > 0

shap_values = shap_linear_importance(X_pre_top[:, rejections], y_pre, scale=False, full=True)
abs_shap = abs(shap_values).mean(axis=0)
abs_shap /= 0.01*abs_shap.sum()
_ = np.argsort(abs_shap)[::-1]
_ = _[abs_shap[_] > 1][:s_max]

rejections = rejections[_]
shap_values = shap_values[:, _]
X_pre_top = X_pre_top[:, rejections]
del _
feature_names[rejections]

In [ ]:
keep, remove = shap_feature_elimination(X_pre_top, y_pre)
X_pre_top = X_pre_top[:, keep]
rejections = rejections[keep]
shap_values = shap_values[:, keep]

alpha = 0.1
classifer_threshold = 0.5
X_non_null = X_pre_top.copy()
shap_non_null = shap_values.copy()
delete_seq = []
n_updates = len(rejections) # 1, len(rejections)
while n_updates > 0:
    stop = True
    Sigma, invSigma = estimate_covariance(X_non_null, shrinkage_tol, shrinkage=shrinkage)
    for j in range(X_non_null.shape[-1]-1, X_non_null.shape[-1]//2, -1):
        classifier_confidences = []
        for _ in trange(50):
            Xk = knockoffs.GaussianSampler(X_non_null, Sigma=Sigma, invSigma=invSigma, 
                                           method='ci').sample_knockoffs()
            Xn = X_non_null.copy()
            Xn[:, j] = Xk[:, j]
            
            swap_shap_values = shap_linear_importance(Xn, y_pre, scale=False, full=True)
            classifier_confidences.append(c2st(shap_non_null[:, j:j+1], swap_shap_values[:, j:j+1], 
                                               clf=linear_model.LogisticRegression(fit_intercept=False)))
    
        classifier_confidences = np.array(classifier_confidences)
        # print("binary classifier's acc:", classifier_confidences.mean())
        pv = stats.wilcoxon(classifier_confidences-classifer_threshold, alternative='greater').pvalue
        
        if not pv < alpha:
            delete_seq.append(j)
            X_non_null = np.delete(X_non_null, j, axis=-1)

            keep, remove = shap_feature_elimination(X_non_null, y_pre)
            X_non_null = X_non_null[:, keep]
            if len(remove) > 0:
                delete_seq.extend(remove)
            print(delete_seq)
            
            shap_non_null = shap_linear_importance(X_non_null, y_pre, scale=False, full=True)
            stop = False
            n_updates -= 1
            break
            
    if stop: break

for j in delete_seq:
    rejections = np.delete(rejections, j)
    X_pre_top = np.delete(X_pre_top, j, axis=-1)
del X_non_null, shap_non_null, delete_seq, keep, remove, Xk, Xn
feature_names[rejections]

In [ ]:
if target_name == 'u':
    assert set(ground_indices_u).issubset(set(rejections))
    print("OK!")
elif target_name == 'v':
    assert set(ground_indices_v).issubset(set(rejections))
    print("OK!")
else:
    assert False

### Best-subset regression

In [ ]:
# X_pre_top = X_pre_top/np.linalg.norm(X_pre_top, 2, axis=0)
# if target_name == 'u':
#     y_pre = uv_pre[:, 0:1]
# elif target_name == 'v':
#     y_pre = uv_pre[:, 1:2]
# y_pre = y_pre/np.linalg.norm(y_pre, 2, axis=0)

X_pre = StandardScaler(with_std=True).fit_transform(X_pre)
rejections = np.union1d(np.where(abs(brute_force(X_pre, y_pre, support_size=len(rejections))) > 0)[0], 
                        rejections)
X_pre_top = X_pre[:, rejections]

# best_subsets = brute_force_all_subsets(normalize_lp(X_pre_top)[0], y_pre)[1]
best_subsets = brute_force_all_subsets(X_pre_top, y_pre)[1]
b_bics = np.array([sm.OLS(y_pre, X_pre_top[:, bs]).fit().bic for bs in best_subsets])
best_subsets = [best_subsets[_] for _ in decreasing_values_indices(b_bics)]
b_bics = b_bics[decreasing_values_indices(b_bics)]

In [ ]:
compromise_programming(best_subsets, (X_pre_top, y_pre))

### Conformal predictions

In [ ]:
# import warnings; warnings.filterwarnings("ignore")
from mapie.utils import train_conformalize_test_split, train_test_split
from mapie.regression import SplitConformalRegressor, JackknifeAfterBootstrapRegressor, CrossConformalRegressor
from mapie.metrics.regression import regression_coverage_score, regression_mean_width_score, regression_mwi_score
from sklearn.metrics import mean_pinball_loss
import uncertainty_toolbox as uct
from model_diagnostics.scoring import decompose, SquaredError, PinballLoss
from icomp import icomp_ifim_complexity, ricomp_m_complexity, ricomp_s_complexity, ricomp_mm_complexity

# Convert (lower, upper) to (mu, sigma)
def interval_to_mu_sigma(lower, upper, confidence_level=0.9):
    mu = (lower + upper) / 2
    z = norm.ppf((1 + confidence_level) / 2)
    sigma = (upper - lower) / (2 * z)
    return mu, sigma

# pinball losses
def pinball_triplet(y_true, y_pred, lower, upper, quantile=0.5, confidence_level=0.9):
    return [mean_pinball_loss(y_true, lower, alpha=1-confidence_level), 
            mean_pinball_loss(y_true, y_pred, alpha=quantile), 
            mean_pinball_loss(y_true, upper, alpha=confidence_level)]

cp_method = 'split' # split, jack, cross
threshold_lambda = 1e6 # 1e4, 1e5, 1e6
conformity_score = "absolute" # "absolute", "residual_normalized" (only valid for SplitConformalRegressor), "gamma"
n_iters = 600
coverage = 0.90
quantile_level = 0.5
ridge_alpha = 1e-5

F = []
coverage_probabilities = []
prediction_widths = []
for bs in best_subsets:
    cp = 0; width = 0
    com = 0; pb = 0
    mc = 0; dc = 0;
    pde_uct = 0; error_bar = 0
    for _ in trange(n_iters):
        ridge = Ridge(fit_intercept=False, alpha=ridge_alpha)

        if cp_method == 'split':
            ## Split conformal regression ##
            (X_train, X_conformalize, X_test, y_train, y_conformalize, y_test) = train_conformalize_test_split(
                X_pre_top[:, bs], y_pre.ravel(), 
                train_size=0.6, conformalize_size=0.2, test_size=0.2, 
                random_state=_
            )
            mapie_regressor = SplitConformalRegressor(estimator=ridge, confidence_level=coverage, conformity_score=conformity_score, prefit=False)
            mapie_regressor.fit(X_train, y_train).conformalize(X_conformalize, y_conformalize)
        else:
            (X_train, X_test, y_train, y_test) = train_test_split(
                X_pre_top[:, bs], y_pre.ravel(), 
                test_size=0.2, 
                random_state=_
            )
            if cp_method == 'jack':
                ## JackknifeAfterBootstrapRegressor ##
                mapie_regressor = JackknifeAfterBootstrapRegressor(estimator=regressor, confidence_level=coverage, conformity_score="absolute", random_state=_)
            elif cp_method == 'cross':
                ## CrossConformalRegressor ##
                mapie_regressor = CrossConformalRegressor(estimator=regressor, confidence_level=coverage, conformity_score="absolute", cv=cv, random_state=_)
            else:
                assert StopIteration
            mapie_regressor.fit_conformalize(X_train, y_train)

        ard = ARDRegression(fit_intercept=False, threshold_lambda=threshold_lambda, max_iter=1000)
        # ard.fit(X_pre_top[:, bs], y_pre.ravel())
        ard.fit(X_train, y_train)
        ard_coef = ard.coef_[abs(ard.coef_) > 0]
        
        ## Predict interval ##
        y_pred, y_pred_interval = mapie_regressor.predict_interval(X_test)
        lower, upper = y_pred_interval[:, 0, 0], y_pred_interval[:, 1, 0]
        y_pred_mu, y_pred_sigma = interval_to_mu_sigma(lower, upper, coverage)
        
        ## metrices ##
        # com += icomp_ifim_complexity(X_train, y_train)
        com += ricomp_mm_complexity(X_train, y_train, update_scale=False) # ricomp_m_complexity, ricomp_s_complexity, ricomp_mm_complexity
        cp += regression_coverage_score(y_test, y_pred_interval)[0]
        width += regression_mean_width_score(y_pred_interval)[0]
        pde_uct += np.linalg.norm(np.sqrt(np.diag(ard.sigma_)), 1)/np.linalg.norm(ard_coef, 1) # PDE uncertainty
        error_bar += np.sum(np.diag(ard.sigma_)/ard_coef**2)
        # uct.metrics_scoring_rule.nll_gaussian(y_pred_mu, y_pred_sigma, y_test, scaled=True)
        # uct.metrics_scoring_rule.check_score(y_test, y_pred_interval, confidence_level=coverage)
        # uct.metrics_scoring_rule.crps_gaussian(y_test, y_pred_interval, confidence_level=coverage)
        # regression_mwi_score(y_test, y_pred_interval, confidence_level=coverage)
        pb += np.mean(pinball_triplet(y_test, y_pred, lower, upper, quantile=quantile_level, confidence_level=coverage))
        miscalibration, discrimination = np.array(decompose(y_test, y_pred, 
                                                            # scoring_function=SquaredError(), 
                                                            scoring_function=PinballLoss(level=quantile_level), 
                                                            functional="quantile")[['miscalibration', 'discrimination']])[0]
        # mc += uct.metrics_calibration.mean_absolute_calibration_error(y_pred_mu, y_pred_sigma, y_test)
        mc += miscalibration
        dc += discrimination

    cp /= n_iters; width /= n_iters
    coverage_probabilities.append(cp)
    prediction_widths.append(width)
    F.append(np.array([com, pb, 
                       pde_uct, error_bar, 
                       mc, dc])/n_iters)
    print("Support size:", len(bs))
    print(cp, width)

F = np.array(F)
(model_complexities, pinball_losses, 
 pde_uncertainties, error_bars, 
 miscalibration, discrimination) = F.T
pde_uncertainties /= pde_uncertainties.min()
complexities = np.array([len(bs) for bs in best_subsets])
error_bars /= error_bars.min()
F = F[~np.isnan(F).any(axis=-1)]

In [ ]:
selected_alternatives = np.sort(np.argsort([sm.OLS(y_pre, X_pre_top[:, bs]).fit().bic for bs in best_subsets])[:5])
# weighting options: 'entropy_weights', 'angle_weights', 'gini_weights', 'variance_weights'
# selected_criteria = [0, 1, 2, 4]; update_weights = True; weighting_method = 'variance_weights'
# selected_criteria = range(len(F.T)); update_weights = False; weighting_method = 'variance_weights'
selected_criteria = [0, 1, 2, 4]; update_weights = False; weighting_method = 'variance_weights'

# preprocessing / normalization
rel0 = lambda _: _ - _.min()
rel1 = lambda _: _ / _.min()
nF = F.copy()
# complexity
nF[:, 0] += complexities
nF[:, 0] = rel0(nF[:, 0])
# pinball_losses
nF[:, 1] = rel0(np.log(nF[:, 1]))
# pde_uncertainty, error_bars
nF[:, 2] = np.log(rel1(nF[:, 2]))
nF[:, 3] = np.log(rel1(nF[:, 3]))
# miscalibration, discrimination
nF[:, 4] = rel0(np.log(nF[:, 4]))
nF[:, 5] = rel0(np.log(nF[:, 5]))

# types (-1 for min / +1 for max) and obj_w
types = np.array([-1, -1, -1, -1, -1, +1])
nF = nF[:, selected_criteria]
obj_weights = getattr(obj_w, weighting_method)(nF, types=types)

# recursive mcdm
keep_from = 0
types = types[selected_criteria]
filtered_F = nF[selected_alternatives].copy()
filtered_complexities = complexities[selected_alternatives].copy()
while len(filtered_F) > 1:
    if update_weights:
        obj_weights = getattr(obj_w, weighting_method)(filtered_F, types=types)
    
    print("Weights:", 100*obj_weights)
    print("Normalized alternatives"); print(filtered_F)
    print()
    filtered_complexities = filtered_complexities[keep_from:]
    ranks, prefs = mcdm(filtered_F, obj_weights, types)
    
    # Rank aggregation ('bd'  -> Borda Method | 'rrf' -> Reciprocal Rank Fusion | 'sc'  -> Schulze Method)
    ranks = np.array(ranks).T.astype(np.int32)
    ranks = rank_aggregation(ranks).run_methods(methods=['bd', 'rrf', 'sc']).values
    ranks = (np.argsort(ranks, axis=0) + 1).T
    borda_counts = np.sum([len(_)+1-_ for _ in ranks], axis=0)
    keep_from = min(np.where(borda_counts == borda_counts.max())[0])
    if keep_from == 0:
        break
    filtered_F = filtered_F[keep_from:]

prefs = np.array(prefs).T
print('>>> Filtered alternatives:', filtered_F)

In [ ]:
with plt.style.context('science'):
    figsize = 4.5; ncols = 4
    fig, (ax1l, ax2l, ax3l, ax4l) = plt.subplots(figsize=(figsize*ncols, ncols), ncols=ncols)
    
    # R2 scores vs Prediction width
    ax1r = ax1l.twinx()
    ax1l.plot(complexities, np.log(pinball_losses), '--o', c='black', markerfacecolor='none', label='Log of the average pinball loss ($-$)')
    ax1l.set_xticks(complexities)
    ax1l.set_xlabel('Support size', fontsize=10.5)
    ax1r.plot(complexities, complexities+model_complexities, '--o', c='tab:blue', markerfacecolor='none', label='Model complexity ($-$)')
    ax1r.tick_params(labelcolor='tab:blue')
    
    # Miscalibration vs Discrimination
    ax2r = ax2l.twinx()
    ax2l.plot(complexities, np.log(miscalibration), '--o', c='black', markerfacecolor='none', label='Log miscalibration ($-$)')
    ax2l.set_xticks(complexities)
    ax2l.set_xlabel('Support size', fontsize=10.5)
    ax2r.plot(complexities, np.log(discrimination), '--o', c='tab:blue', markerfacecolor='none', label='Log discrimination ($+$)')
    ax2r.tick_params(labelcolor='tab:blue')

    # Winkler interval score vs Continuous ranked probability score
    ax3r = ax3l.twinx()
    ax3l.plot(complexities, np.log(pde_uncertainties), '--o', c='black', markerfacecolor='none', label='Log PDE uncertainty ($-$)')
    ax3l.set_xticks(complexities)
    ax3l.set_xlabel('Support size', fontsize=10.5)
    ax3r.plot(complexities, np.log(error_bars), '--o', c='tab:blue', markerfacecolor='none', label='Log error bar ($-$)')
    ax3r.tick_params(labelcolor='tab:blue')

    # pde_uncertainty, error_bars
    ax4r = ax4l.twinx()
    ax4l.plot(filtered_complexities, prefs, '-o', label=["TOPSIS ($+$)", "MABAC ($+$)", "COMET ($+$)", "SPOTIS ($-$)"])
    ax4l.set_xticks(filtered_complexities)
    ax4l.set_ylabel("Preference ($+$) / Dispreference ($-$)", fontsize=10.5)
    ax4l.set_xlabel("Support size", fontsize=10.5)
    
    for _, axlr in enumerate([[ax1l, ax1r], [ax2l, ax2r], [ax3l, ax3r], [ax4l, ax4r]]):
        handles = []
        labels = []
        for ax in axlr:
            h, l = ax.get_legend_handles_labels()
            handles.extend(h)
            labels.extend(l)
        fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(1/ncols*_+0.125, 1.11), ncol=len(labels)//2, fontsize=10.5,
                   frameon=True, fancybox=True, shadow=True)
    
    fig.tight_layout()
    plt.show()

In [ ]:
tau = 3
verbose = True
# scale = 1 <- generalized UBIC
scale = np.log(len(y_pre))
per = 75 # 80

post_means, b_bics, b_uns = baye_uncertainties(best_subsets, (X_pre_top, y_pre), 
                                               u_type='cv1', take_sqrt=True, 
                                               ridge_lambda=0, 
                                               threshold=0)
# b_uns = ard_uns # USE ard_uns INSTEAD
predictions = X_pre_top@post_means
print(b_bics)
print(b_uns)
b_bics = np.array(b_bics)
max_complexity = len(b_bics)
complexities = np.arange(max_complexity)+1
d_complexities = complexities[decreasing_values_indices(b_bics)]
d_bics = b_bics[decreasing_values_indices(b_bics)]
slopes = np.diff(d_bics)/(np.diff(d_complexities)*d_bics[:-1])
try:
    thres = np.percentile(np.abs(slopes), per)
    # None / Round / Ceil / Floor: Decided by researchers to automate the model selection process
    thres = np.round(sci_format(thres)[0])*10**sci_format(thres)[1]
except IndexError:
    thres = 1/40
min_thres = 1/40; max_thres = 1/10
thres = min(max(thres, min_thres), min_thres)
print("threshold:", thres)

lower_bounds = []
for k, efi in enumerate(best_subsets):
    # assert len(efi) == np.count_nonzero(post_means[:, k:k+1])
    com = len(efi)
    lower_bound = 2*np.abs(log_like_value(predictions[:, k:k+1], y_pre))-np.log(len(y_pre))*com
    lower_bounds.append(lower_bound)

last_lam = np.log10(max(lower_bounds/(b_uns*scale)))
print("max_lam:", last_lam)
delta = last_lam/tau
now_lam = last_lam-delta
last_ubic = UBIC(b_bics, b_uns, len(y_pre), hyp=10**last_lam, scale=scale)
last_bc = np.argmin(last_ubic)
bc_seq = [last_bc]
while now_lam >= 0:
    now_ubic = UBIC(b_bics, b_uns, len(y_pre), hyp=10**now_lam, scale=scale)
    now_bc = np.argmin(now_ubic)
    
    diff_com = now_bc-last_bc
    diff_bic = b_bics[now_bc]-b_bics[last_bc]
    imp = np.nan
    if diff_com != 0:
        imp = abs(diff_bic/(b_bics[last_bc]*diff_com))
    
    if verbose:
        print(min(last_bc, now_bc), '<--->', max(last_bc, now_bc), 
              np.nan_to_num(imp, nan=np.inf))
    
    if (diff_com > 0 and (diff_bic > 0 or imp < thres)) or \
        (diff_com < 0 and diff_bic > 0 and imp > thres):
        break
    
    last_lam = now_lam
    now_lam = round(last_lam-delta, 8)
    last_ubic = now_ubic
    last_bc = now_bc
    if last_bc not in bc_seq:
        bc_seq.append(last_bc)

# best_bc = knee(range(len(last_ubic)), last_ubic, 0.95, 'linear', direction='decreasing')
best_bc = knee_finder(last_ubic)
if best_bc == 0 and last_bc != 0 and b_bics[last_bc] < b_bics[0] and \
                                    abs((b_bics[last_bc]-b_bics[0])/(b_bics[0]*last_bc)) > thres:
    best_bc = knee(range(1, len(last_ubic)), last_ubic[1:], 0.95, 'linear')
if best_bc < last_bc and abs((b_bics[last_bc]-b_bics[best_bc])/(b_bics[best_bc]*(last_bc-best_bc))) > thres:
    best_bc = last_bc
    
last_lam = round(last_lam, 8)
last_lam, last_ubic, last_bc, best_bc

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot([len(bs) for bs in best_subsets], last_ubic, '-o', c='black')
plt.show()

In [ ]:
_, best_subsets = brute_force_all_subsets(X_pre_top, y_pre, max_support_size=8)
ebics = []
mbics = []
for _ in best_subsets:
    loglik = log_like_value(X_pre_top[:, _]@np.linalg.lstsq(X_pre_top[:, _], y_pre, rcond=None)[0], 
                            y_pre)
    ebics.append(ebic(loglik, len(_), len(y_pre), X_pre_top.shape[-1], const=0))
    mbics.append(mbic(loglik, len(_), len(y_pre), X_pre_top.shape[-1], const=2))
ebics = np.array(ebics)
mbics = np.array(mbics)

In [ ]:
# Assume that mbics is a decreasing sequence
complexities = np.array([len(_) for _ in best_subsets])
if np.alltrue(np.array(mbics) >= np.array([max(mbics)+_*(min(mbics)-max(mbics))/(np.argmin(mbics)-np.argmax(mbics)) for _ in range(len(best_subsets))])):
    knee = complexities.max()
else:
    decreasing_indices = np.array(mbics) <= np.array([max(mbics)+_*(min(mbics)-max(mbics))/(np.argmin(mbics)-np.argmax(mbics)) for _ in range(len(best_subsets))])
    knee = knee_finder(mbics[decreasing_indices])
    knee = (complexities[decreasing_indices])[knee]
knee

In [ ]:
SEED = 1234
np.random.seed(SEED); random.seed(SEED)
n_samples = int(250*knee)
false_discovery_control_method = None
fdr_data = []
for bs in best_subsets:
    fdrs = []
    for _ in range(len(y_pre)//n_samples):
        X_test = X_pre_top[:, bs]
        y_test = y_pre.ravel()
        
        np.random.seed(random.randint(0, 100))
        # sample_indices = sorted(set([np.random.randint(len(y_pre)) for _ in range(n_samples)]))
        sample_indices = fpsample.bucket_fps_kdline_sampling(X_test, n_samples=n_samples, h=3) # Farthest Point Sampling (FPS) is better!!!
        X_test = X_test[sample_indices]; y_test = y_test[sample_indices]
        # FPS + k-DPP
        DPP = FiniteDPP('likelihood', **{'L': X_test.dot(X_test.T)})
        DPP.flush_samples()
        for _ in range(n_samples//(len(bs))):
            DPP.sample_exact_k_dpp(size=len(bs))
        sample_indices = np.unique(np.ravel(DPP.list_of_samples))
        X_test = X_test[sample_indices]; y_test = y_test[sample_indices]
        
        manager = stepwise_selective_inference(support_size=X_test.shape[1])
        M, p_list = manager.inference(X_test, y_test, np.std(y_test))
        if false_discovery_control_method is not None:
            p_list = stats.false_discovery_control(p_list, method=false_discovery_control_method)
            
        fdrs.append(subset_fdr(p_list))
        
    fdrs = np.array(fdrs)
    if fdrs.mean() < 1:
        print(len(bs), fdrs.mean())
        fdr_data.append(fdrs)
        
fdr_data = np.array(fdr_data)

from sklearn.cluster import AffinityPropagation, KMeans
print(AffinityPropagation().fit(fdr_data).labels_)
print(KMeans(n_clusters=2).fit(fdr_data).labels_)

In [ ]:
SEED = 1234
np.random.seed(SEED); random.seed(SEED)
n_samples = int(250*knee)
false_discovery_control_method = 'by'
fdr_data = []
for bs in best_subsets:
    fdrs = []
    for _ in range(len(y_pre)//n_samples):
        X_test = X_pre_top[:, bs]
        y_test = y_pre.ravel()
        
        np.random.seed(random.randint(0, 100))
        # sample_indices = sorted(set([np.random.randint(len(y_pre)) for _ in range(n_samples)]))
        sample_indices = fpsample.bucket_fps_kdline_sampling(X_test, n_samples=n_samples, h=3) # Farthest Point Sampling (FPS) is better!!!
        X_test = X_test[sample_indices]; y_test = y_test[sample_indices]
        # FPS + k-DPP
        DPP = FiniteDPP('likelihood', **{'L': X_test.dot(X_test.T)})
        DPP.flush_samples()
        for _ in range(n_samples//(len(bs))):
            DPP.sample_exact_k_dpp(size=len(bs))
        sample_indices = np.unique(np.ravel(DPP.list_of_samples))
        X_test = X_test[sample_indices]; y_test = y_test[sample_indices]
        
        manager = stepwise_selective_inference(support_size=X_test.shape[1])
        M, p_list = manager.inference(X_test, y_test, np.std(y_test))
        if false_discovery_control_method is not None:
            p_list = stats.false_discovery_control(p_list, method=false_discovery_control_method)
            
        fdrs.append(subset_fdr(p_list))
        
    fdrs = np.array(fdrs)
    if fdrs.mean() < 1:
        print(len(bs), fdrs.mean())
        fdr_data.append(fdrs)
        
fdr_data = np.array(fdr_data)

### Selective inference

### Python

In [ ]:
n_terms = 16
max_complexity = 14
alphas = [0.3, 0.25, 0.2, 0.15, 0.1, 0.05, 0.01]

# Choose alpha_min between 1e-6 and 1e-7, 
# if 1e-7 results in a higher number of rejections but does not compromise max_fdr (not making it less strict)
_, lars_p, _ = lars_path(StandardScaler().fit_transform(X_pre), y_pre.flatten(), method='lasso', alpha_min=1e-7)
lars_p = np.array(list(map(int, lars_p)))[:n_terms]

# nonzero = np.nonzero(AbessLinearRegression(s_min=1, path_type='gs', fit_intercept=False, alpha=1e-9, max_iter=100).fit(X_pre, y_pre.flatten()).coef_)[0]
# nonzero = np.nonzero(MIOSR(X_pre, y_pre, alpha=1e-9, non_zero=min(len(nonzero), n_terms)))[0]
# _, lars_p, _ = lars_path(StandardScaler().fit_transform(X_pre[:, nonzero]), y_pre.flatten(), method='lasso', alpha_min=0)
# lars_p = nonzero[np.array(list(map(int, lars_p)))][:n_terms]

X_test = X_pre[:, lars_p]
sigma = np.std(y_pre-X_test@np.linalg.lstsq(X_test, y_pre)[0], ddof=1)
manager = stepwise_selective_inference(support_size=len(lars_p))
_, p_list = manager.inference(X_test, y_pre, sigma)
print(lars_p, p_list, subset_fdr(p_list))

for alpha in alphas:
    adjusted_pvalues = p_list
    stop_step, false_discovery_rates = forward_stop_rule(adjusted_pvalues, alpha)
    adjusted_pvalues = adjusted_pvalues[:stop_step+1]
    rejections = np.sort(lars_p[:stop_step+1])
    if len(rejections) <= max_complexity: 
        break
max_fdr = alpha
max_fdr, feature_names[rejections]

### R

In [ ]:
# max_complexity = 14
# alphas = [0.3, 0.2, 0.1, 0.05, 0.01]
# for alpha in alphas:
#     adjusted_pvalues = fsInf.get("pv")
#     stop_step, false_discovery_rates = forward_stop_rule(adjusted_pvalues, alpha)
#     adjusted_pvalues = adjusted_pvalues[:stop_step+1]
#     rejections = np.sort((fsInf.get("vars")-1).astype(np.int32)[:stop_step+1])
#     if len(rejections) <= max_complexity:
#         break
# max_fdr = alpha
# feature_names[rejections]

In [ ]:
X_pre_top = X_pre[:, rejections]
X_pre_top = X_pre_top/np.linalg.norm(X_pre_top, 2, axis=0)

In [ ]:
_, best_subsets = brute_force_all_subsets(X_pre_top, y_pre, max_support_size=8)
ebics = []
mbics = []
for _ in best_subsets:
    loglik = log_like_value(X_pre_top[:, _]@np.linalg.lstsq(X_pre_top[:, _], y_pre, rcond=None)[0], 
                            y_pre)
    ebics.append(ebic(loglik, len(_), len(y_pre), X_pre_top.shape[-1], const=0))
    mbics.append(mbic(loglik, len(_), len(y_pre), X_pre_top.shape[-1], const=2))
ebics = np.array(ebics)
mbics = np.array(mbics)

In [ ]:
# Assume that mbics is a decreasing sequence
complexities = np.array([len(_) for _ in best_subsets])
if np.alltrue(np.array(mbics) >= np.array([max(mbics)+_*(min(mbics)-max(mbics))/(np.argmin(mbics)-np.argmax(mbics)) for _ in range(len(best_subsets))])):
    knee = complexities.max()
else:
    decreasing_indices = np.array(mbics) <= np.array([max(mbics)+_*(min(mbics)-max(mbics))/(np.argmin(mbics)-np.argmax(mbics)) for _ in range(len(best_subsets))])
    knee = knee_finder(mbics[decreasing_indices])
    knee = (complexities[decreasing_indices])[knee]
knee

In [ ]:
SEED = 1234
np.random.seed(SEED); random.seed(SEED)
n_samples = int(250*knee)
max_fdr = alpha; false_discovery_control_method = None
for bs in best_subsets:
    fdrs = []
    for _ in range(len(y_pre)//n_samples):
        X_test = X_pre_top[:, bs]
        y_test = y_pre.ravel()
        
        np.random.seed(random.randint(0, 100))
        # sample_indices = sorted(set([np.random.randint(len(y_pre)) for _ in range(n_samples)]))
        sample_indices = fpsample.bucket_fps_kdline_sampling(X_test, n_samples=n_samples, h=3) # Farthest Point Sampling (FPS) is better!!!
        X_test = X_test[sample_indices]; y_test = y_test[sample_indices]
        # FPS + k-DPP
        DPP = FiniteDPP('likelihood', **{'L': X_test.dot(X_test.T)})
        DPP.flush_samples()
        for _ in range(n_samples//(len(bs))):
            DPP.sample_exact_k_dpp(size=len(bs))
        sample_indices = np.unique(np.ravel(DPP.list_of_samples))
        X_test = X_test[sample_indices]; y_test = y_test[sample_indices]
        
        manager = stepwise_selective_inference(support_size=X_test.shape[1])
        M, p_list = manager.inference(X_test, y_test, np.std(y_test))
        if false_discovery_control_method is not None:
            p_list = stats.false_discovery_control(p_list, method=false_discovery_control_method)
        # print(M, p_list, np.array(p_list) < 0.05)
        fdrs.append(subset_fdr(p_list))
        
    fdrs = np.array(fdrs)
    print(fdrs.mean(), stats.wilcoxon(fdrs-max_fdr, alternative='less').pvalue)